# LAB — Troubleshooting a Broken Job

**Goal:** find and fix the faults in a broken 3-task Lakeflow Job. You read the run history, the task DAG,
the error details and the Spark UI or query profile metrics. These are the skills of
**exam domain 6 (Troubleshooting, Monitoring & Optimization)**.

**Scenario:** on Friday a teammate refactored the RetailHub nightly job **`retailhub_broken_job`**
(`ingest → transform → publish`). Since then the job fails every night. The trainer deployed it, and its
sources are in `materials/troubleshooting/broken_job/`. You have view access to the job, so you diagnose
the faults in the run UI. Then you fix scratch copies of the tasks in this notebook. They write `ts_*`
tables to your own catalog.

**Prerequisites:**
- The trainer has deployed and run `retailhub_broken_job`, and you have CAN VIEW on it. You do not need the CLI.

| Task | What you do | Where |
|------|-------------|-------|
| 1 | Read the failure in the run history and the DAG, and record the diagnosis | UI + notebook |
| 2 | Fix the ingest path and schema in a scratch copy and show that it runs | notebook |
| 3 | Find the cause of the row explosion in the transform join from query profile metrics, and fix it | notebook |
| 4 | Right-size `spark.sql.shuffle.partitions` | notebook |
| 5 | Fix the publish task, which cannot read its source table | notebook |
| 6 | Bonus: cluster startup failures and library conflicts | notebook |

> **SOLUTION NOTEBOOK:** full answers for `lab_troubleshooting`. It runs top to bottom on a fresh
> participant catalog. It never runs the broken job, only the fixed scratch copies.

## Setup

Run both cells. The second one sets the names of the scratch tables and the source files of this lab.

In [0]:
%run ../setup/00_setup

In [0]:
# Name the scratch tables (ts_ prefix, in YOUR catalog) and the source files
RAW_TABLE      = f"{CATALOG}.{BRONZE_SCHEMA}.ts_orders_raw"
ENRICHED_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.ts_orders_enriched"
GOLD_TABLE     = f"{CATALOG}.{GOLD_SCHEMA}.ts_daily_sales"

ORDERS_JSON    = f"{DATASET_PATH}/orders/orders_batch.json"
CUSTOMERS_CSV  = f"{DATASET_PATH}/customers/customers.csv"

print(RAW_TABLE, ENRICHED_TABLE, GOLD_TABLE, sep="\n")

## Task 1 — Read the failure from the run history

When a job fails, start with its run history. It shows which task failed first and why.
Open **Jobs & Pipelines → `retailhub_broken_job` → latest run**. Look at the **Graph** of the run and
the error of the red task, then fill in `diagnosis_1`.

**What you need to do:**
1. In the **Graph**, find the red task. Note the state of the two tasks after it.
2. Note which task `publish` waits for: the arrow into `publish` in the **Graph**, or the **Depends on** column in the **List** view. Task 5 needs this edge.
3. Click the red task. The error is the red banner at the top of the task page: read its first line.
4. Compare the failing path with the real Volume layout. `00_setup` printed the working path.

The red task shows **2 attempts**: serverless jobs retry a failed task once by themselves (*Retry: 1st*). It is still one run, and both attempts fail with the same error.

The next cell runs the same read with the broken path, so you can also read the error here.

In [0]:
# Reproduce the ingest error with the broken path (read-only, safe to run).
# This is the exact path from materials/troubleshooting/broken_job/task_ingest.py:
BROKEN_PATH = f"/Volumes/{CATALOG}/default/dataset/orders/orders_batch.json"

try:
    spark.read.format("json").load(BROKEN_PATH).count()
    print("Unexpectedly worked?!")
except Exception as e:
    print(f"Error type : {type(e).__name__}")
    print(f"First line : {str(e).splitlines()[0]}")
print(f"\nWorking base path from 00_setup: {DATASET_PATH}")

In [0]:
# Map what the UI shows to the allowed values:
#   - downstream tasks labelled "Upstream failed" in the DAG   -> "skipped" (they never ran)
#   - an error such as [UC_VOLUME_NOT_FOUND] or [PATH_NOT_FOUND] -> "path_not_found"
diagnosis_1 = {
    "failed_task":            "ingest",
    "downstream_task_state":  "skipped",   # shown as skipped / "Upstream failed"
    "error_category":         "path_not_found",
    "wrong_path_segment":     "dataset",   # the Volume is named "datasets"
    "publish_depends_on":     "ingest",    # the Graph: the arrow into publish comes from ingest
}
print(diagnosis_1)

#### Hint — Task 1

Read the run in this order: run history, DAG, error. The run list shows when the failures started
(every run since Friday). The **Graph** shows where: the first red task. The red banner on the task page shows why.

In the **Graph**, a task marked *Upstream failed* is a skipped task. It did not run at all, so do not debug it
yet. Find the cause of the first failure, because the states of the later tasks follow from it.

The first line of the error carries the error class, for example
`[UC_VOLUME_NOT_FOUND] Volume <catalog>.default.<name> does not exist`. The path segment right after
`default/` is read as the Volume name. A wrong folder inside an existing Volume raises `[PATH_NOT_FOUND]`
instead. Compare the failing path segment by segment with a path that works (`DATASET_PATH`).

In [0]:
# -- Validation --
assert diagnosis_1["failed_task"] == "ingest", "Which node is red FIRST in the DAG?"
assert diagnosis_1["downstream_task_state"] == "skipped", \
    "Tasks behind a failed dependency never run — check their state label"
assert diagnosis_1["error_category"] == "path_not_found", "Re-read the first line of the error"
assert str(diagnosis_1["wrong_path_segment"]).lower().strip("/") == "dataset", \
    "Compare the broken path with DATASET_PATH segment by segment"
assert diagnosis_1["publish_depends_on"] == "ingest", \
    "Open the Graph of the run (or List -> Depends on): which task does publish wait for?"
print("Task 1 OK: root cause located in the ingest task (bad Volume path)")

## Task 2 — Fix the path and the schema in a scratch copy

The path is not the only fault in the ingest task. `task_ingest.py` also declares this reader schema:

```python
orders_schema = StructType([
    StructField("order_id",       StringType()),
    StructField("client_id",      StringType()),   # ???
    StructField("product_id",     StringType()),
    StructField("store_id",       StringType()),
    StructField("order_datetime", StringType()),
    StructField("quantity",       IntegerType()),
    StructField("unit_price",     DoubleType()),
    StructField("amount_total",   DoubleType()),   # ???
    StructField("payment_method", StringType()),
])
```

With a user-supplied schema, a schema field that matches no JSON field is filled with NULL, and the read
raises no error. So after you fix the path, the task fails on its quality gate: *"N/N rows have NULL
client_id/amount_total"*. Check which field names the source file really uses.

**What you need to do:** write the corrected ingest into `RAW_TABLE`. Use the correct path (`ORDERS_JSON`)
and the corrected field names.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

fixed_schema = StructType([
    StructField("order_id",       StringType()),
    StructField("customer_id",    StringType()),   # was: client_id
    StructField("product_id",     StringType()),
    StructField("store_id",       StringType()),
    StructField("order_datetime", StringType()),
    StructField("quantity",       IntegerType()),
    StructField("unit_price",     DoubleType()),
    StructField("total_amount",   DoubleType()),   # was: amount_total
    StructField("payment_method", StringType()),
])

fixed_orders_df = (
    spark.read
    .format("json")
    .schema(fixed_schema)
    .load(ORDERS_JSON)
)

fixed_orders_df.write.mode("overwrite").saveAsTable(RAW_TABLE)
print(f"Written {RAW_TABLE}: {spark.table(RAW_TABLE).count():,} rows")

#### Hint — Task 2

The read raised no error because a JSON reader with an explicit schema looks up each field by name.
A schema column that matches no JSON field is filled with NULL, with no exception and no warning.
That is why quality gates (or `rescued data` in Auto Loader) matter at Bronze.

To find the two wrong names, print the start of the file and compare its field names with the schema:

```python
print(dbutils.fs.head(ORDERS_JSON, 300))
```

The reader call has this shape. The `write` line is already in the cell.

```python
fixed_orders_df = spark.read.format("<format>").schema(<schema>).load(<path>)
```

In [0]:
# -- Validation --
# This is the broken task's own quality gate — now it must pass
_raw   = spark.table(RAW_TABLE)
_total = _raw.count()
assert _total > 0, "RAW_TABLE is empty"
assert "customer_id" in _raw.columns and "total_amount" in _raw.columns, \
    f"Schema still wrong: {_raw.columns}"
_null_keys = _raw.filter("customer_id IS NULL OR total_amount IS NULL").count()
assert _null_keys < _total, "All keys NULL — a schema field name still doesn't match the JSON"
# ~3% of the source rows have a NULL customer_id / total_amount on purpose (dirty data), so allow up to 5%;
# a field name that still does not match the JSON makes that column NULL in 100% of rows.
assert _null_keys / _total < 0.05, f"{_null_keys}/{_total} NULL keys — check both corrected names"
print(f"Task 2 OK: quality gate passed — {_total - _null_keys:,}/{_total:,} rows populated")

## Task 3 — Diagnose the row-exploding join

With ingest fixed, `transform` succeeds, but it writes about 100 times more rows than it reads.
An enrichment join adds customer columns to each order, so it must not multiply the orders.

This is how the problem looks in the Spark UI or the serverless query profile:

| Metric (join stage) | Healthy enrichment join | This job |
|---|---|---|
| Output rows vs input rows | ≈ input (1:1) | **~100× input** |
| Shuffle write | small | inflated by the duplicate rows |
| Downstream write task time | seconds | minutes, all in the write after the join |

First reproduce the join and measure it. Then fill in the diagnosis and fix the join.

**What you need to do:** count the distinct truncated keys in `distinct_bad_keys`, fill in `diagnosis_3`, then write the fixed join to `ENRICHED_TABLE`.

In [0]:
from pyspark.sql import functions as F

orders    = spark.table(RAW_TABLE)
customers = (spark.read.format("csv").option("header", True).load(CUSTOMERS_CSV))

orders_count = orders.count()

# The teammate's join, verbatim from task_transform.py:
exploded = orders.join(
    customers,
    F.substring(orders["customer_id"], 1, 8) == F.substring(customers["customer_id"], 1, 8),
    "inner",
)
exploded_count = exploded.count()

print(f"orders in    : {orders_count:>12,}")
print(f"joined out   : {exploded_count:>12,}   <- explosion factor ~{exploded_count/orders_count:.0f}x")
print("\nServerless: click 'See performance' under this cell's output → Query profile → join node → rows.")

In [0]:
distinct_bad_keys = (
    customers.select(F.substring("customer_id", 1, 8).alias("k")).distinct().count()
)

diagnosis_3 = {
    "symptom_metric":  "output_rows",
    "root_cause":      "truncated_low_cardinality_join_key",
    "fix":             "join_on_full_customer_id",
}
print(f"distinct truncated keys: {distinct_bad_keys}")
print(diagnosis_3)

In [0]:
enriched_fixed = orders.join(
    customers,
    orders["customer_id"] == customers["customer_id"],
    "inner",
).select(
    orders["*"],
    customers["first_name"],
    customers["last_name"],
    customers["city"],
    customers["country"],
    customers["customer_segment"],
)

enriched_fixed.write.mode("overwrite").saveAsTable(ENRICHED_TABLE)
print(f"{ENRICHED_TABLE}: {spark.table(ENRICHED_TABLE).count():,} rows")

#### Hint — Task 3

On classic compute, open the Spark UI, go to the **SQL/DataFrame** tab and open the join query. Compare
the *output rows* of the SortMergeJoin or ShuffleHashJoin node with its inputs. Then check the shuffle read
and write sizes in **Stages**. On **serverless**, click **See performance** under the cell output, open
**Query profile** and select the join node. It shows the same row counts.

Check the cardinality of the join key with data. Count the distinct values of the expression the join
really uses: `F.substring(col, 1, 8)` builds the truncated key, and `.select(...).distinct().count()` counts
its values. Compare that number with the number of customers.

Before you choose a fix, ask whether more compute would change the number of output rows. Then read the
join condition again and look for the column that identifies exactly one customer.

In [0]:
# -- Validation --
assert 50 <= distinct_bad_keys <= 200, \
    f"Expected ~100 distinct truncated keys, got {distinct_bad_keys} — count left(customer_id, 8) over customers"
assert diagnosis_3["symptom_metric"] == "output_rows", "The smoking gun is the join node's output rows"
assert diagnosis_3["root_cause"] == "truncated_low_cardinality_join_key", "Re-check the join condition"
assert diagnosis_3["fix"] == "join_on_full_customer_id", "The fix is the key, not the cluster"

_enriched_count = spark.table(ENRICHED_TABLE).count()
assert _enriched_count < exploded_count / 10, \
    f"Still exploding: {_enriched_count:,} rows — is the join on the FULL customer_id?"
assert _enriched_count <= orders_count, "An enrichment inner join must not multiply orders"
assert _enriched_count >= orders_count * 0.9, \
    f"Lost too many rows ({_enriched_count:,} of {orders_count:,}) — wrong join type or key?"
print(f"Task 3 OK: {_enriched_count:,} rows (was {exploded_count:,} exploded)")

## Task 4 — Right-size the shuffle partitions

`task_transform.py` also sets:

```python
spark.conf.set("spark.sql.shuffle.partitions", "4000")
```

The teammate copied this value from a blog post about a 50 TB workload. On a classic cluster, a shuffle of a
few hundred MB split into 4000 partitions runs 4000 tasks of a few milliseconds each, and scheduling them takes
longer than the work itself.

On serverless you will not see this effect here. This table is small (about 100,000 rows, 2 MB), and serverless
coalesces the shuffle partitions by itself, so both runs below take under a second, and the query profile may
show no Shuffle node at all. The timing will not show the problem, so answer from the classic case described above: 4000 tasks of a few milliseconds, where scheduling costs more than the work.
The tiny-task pattern shows on classic compute, in **Spark UI → Stages** (task count).

**What you need to do:** run the query with 4000 partitions, set a better value, run it again and fill in
`answer_4`.

In [0]:
import time

def timed_groupby():
    t0 = time.time()
    (spark.table(ENRICHED_TABLE)
        .groupBy("country", "payment_method")
        .count()
        .collect())
    return time.time() - t0

spark.conf.set("spark.sql.shuffle.partitions", "4000")   # the teammate's setting
t_bad = timed_groupby()
print(f"shuffle.partitions=4000 : {t_bad:.1f}s")

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")

t_good = timed_groupby()
print(f"shuffle.partitions={spark.conf.get('spark.sql.shuffle.partitions')} : {t_good:.1f}s (was {t_bad:.1f}s)")

answer_4 = {
    "why_4000_is_bad": "tiny_task_scheduling_overhead",
    "better_value":    "auto",
}
print(answer_4)

#### Hint — Task 4

On classic compute, open the Spark UI, **Stages** tab, and find the shuffle stage. Look at the **task count**
(it follows `spark.sql.shuffle.partitions` unless AQE coalesces the partitions) and the median task duration.
On serverless the query profile of this small table shows no such stage, so answer from the classic case. Compare *task count × median time*
with the wall-clock time of the stage. If most tasks take only a few *milliseconds*, ask what the stage
spends its time on.

On serverless and on any DBR with AQE enabled, AQE can coalesce the post-shuffle partitions to a size
that fits the data. Find the value of this setting that turns on auto-optimized shuffle, which picks the partition count from the query plan and the input size.

For manual tuning on classic clusters, the usual target is about **128–200 MB per shuffle partition**,
that is `shuffle data size / 128MB`. Work out what that gives for a few hundred MB and compare it with 4000.

`spark.sql.shuffle.partitions` is one of the few Spark confs you can still set on serverless.

In [0]:
# -- Validation --
assert answer_4["why_4000_is_bad"] == "tiny_task_scheduling_overhead", \
    "4000 partitions over ~hundreds of MB = thousands of ms-long tasks"
_bv = answer_4["better_value"]
assert _bv == "auto" or (isinstance(_bv, int) and 1 <= _bv <= 200), \
    "Use 'auto' (AQE/serverless) or a small integer sized at ~128MB per partition"
_current = str(spark.conf.get("spark.sql.shuffle.partitions"))
assert _current != "4000", "The session still runs with 4000 — set your better_value for real"
print(f"Task 4 OK: shuffle.partitions={_current}")

## Task 5 — Fix the broken dependency

The last task, `publish`, has two wiring problems. One is in its code and one is in the job definition.
Look at both before you change anything.

The table it reads, from `task_publish.py`:

```python
SOURCE = f"{CATALOG}.silver.ts_orders_final"
```

Its edge in the job: you recorded it in Task 1 (`diagnosis_1["publish_depends_on"]`) from the **Graph** of the run.

**What you need to do:** reproduce the error, fill in `diagnosis_5`, then run a corrected publish
into `GOLD_TABLE`.

> You cannot edit the job, because you have view access only. Record the fix in `diagnosis_5`, and the
> trainer applies it to the job.

In [0]:
# Reproduce the publish failure (read-only)
try:
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.ts_orders_final").count()
except Exception as e:
    print(f"Error type : {type(e).__name__}")
    print(f"First line : {str(e).splitlines()[0]}")

print("\nTables that DO exist in silver:")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SILVER_SCHEMA} LIKE 'ts_*'"))

In [0]:
diagnosis_5 = {
    "missing_table":           "ts_orders_final",
    "table_actually_produced": "ts_orders_enriched",
    "dag_fix":                 "publish_depends_on_transform",
}
print(diagnosis_5)

from pyspark.sql import functions as F
daily_sales_fixed = (
    spark.table(ENRICHED_TABLE)
    .groupBy(F.to_date("order_datetime").alias("order_date"), "country")
    .agg(
        F.count("*").alias("order_count"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value"),
    )
)
daily_sales_fixed.write.mode("overwrite").saveAsTable(GOLD_TABLE)
print(f"{GOLD_TABLE}: {spark.table(GOLD_TABLE).count():,} rows")

#### Hint — Task 5

In a multi-task job, `TABLE_OR_VIEW_NOT_FOUND` usually has one of three causes. Check each of them:
1. The producing task writes a different name than the consumer reads. Compare `SOURCE` with the tables
   that the `SHOW TABLES` output above lists.
2. The consumer runs before the producer, because a `depends_on` edge is missing. Find the task that writes
   the table publish needs, and check whether publish waits for it.
3. The catalog or schema context is wrong.

Check the edge even after the name is right. With a wrong edge, the job fails only on the runs where
publish starts before its source table is written.

The corrected aggregation (as in `task_publish.py`) produces these columns:

| Output column | Expression |
|---|---|
| `order_date` | group key: `F.to_date("order_datetime")` |
| `country` | group key |
| `order_count` | `F.count("*")` |
| `revenue` | `F.round(F.sum("total_amount"), 2)` |
| `avg_order_value` | `F.round(F.avg("total_amount"), 2)` |

In [0]:
# -- Validation --
assert str(diagnosis_5["missing_table"]).lower() == "ts_orders_final"
assert str(diagnosis_5["table_actually_produced"]).lower() == "ts_orders_enriched"
assert diagnosis_5["dag_fix"] == "publish_depends_on_transform", \
    "publish consumes transform's output — the DAG edge must say so"

_gold = spark.table(GOLD_TABLE)
assert _gold.count() > 0, "GOLD_TABLE is empty"
for _c in ["order_date", "country", "order_count", "revenue", "avg_order_value"]:
    assert _c in _gold.columns, f"Missing column {_c} in {GOLD_TABLE}: {_gold.columns}"
print(f"Task 5 OK: publish fixed — {_gold.count():,} daily rows in {GOLD_TABLE}")

## Task 6 — Bonus: Failures that happen *before* your code runs

Some failures happen before any of your code runs. They come up on the exam, but you cannot reproduce
them in a notebook. Read the two scenarios and fill in `bonus_6`.

**Scenario A: cluster startup failure.** On Monday at 06:00 the nightly job (classic cluster) fails after
0 seconds of task time with *"Cluster terminated. Reason: Init script failure"*. No notebook cell ran.
- No code ran, so where do you look first?
- Typical causes are an init script with a non-zero exit code, missing instance capacity and a driver OOM at start.

**Scenario B: library conflict.** Someone added `pandas==2.2` as a *cluster* library. Since then an
unrelated job on the same cluster fails at import with `AttributeError`- or `ClassNotFoundException`-style errors.
- What is the cleanest way to give one job its own library versions without changing the cluster?

In [0]:
bonus_6 = {
    "startup_failure_first_look": "cluster_event_log",   # Compute -> cluster -> Event log (+ init script log)
    "library_conflict_fix":       "notebook_scoped_pip", # %pip install pandas==2.2 in THAT job's notebook, pinned
}
print(bonus_6)

In [0]:
# -- Validation --
assert bonus_6["startup_failure_first_look"] == "cluster_event_log", \
    "Startup failures happen before user code — Compute -> Event log (init script log)"
assert bonus_6["library_conflict_fix"] == "notebook_scoped_pip", \
    "%pip installs are scoped to one session — no cluster-wide blast radius"
print("Task 6 OK")

## Summary — mapping to exam domain 6 (Troubleshooting, Monitoring & Optimization)

| Task | What you practiced | Domain 6 objective |
|------|--------------------|--------------------|
| 1 | Run history, the Graph of the run, first red task, first line of the error | Diagnose job failures from run details; tell failed tasks from skipped (upstream failed) tasks |
| 2 | Silent NULLs from a mismatched user schema; quality gates | Find the cause of schema mismatches and bad paths; explain why explicit-schema reads fail without an error |
| 3 | Join node output rows, shuffle write, cardinality checks | Use Spark UI and query profile metrics to find row explosion and skew-like symptoms |
| 4 | `spark.sql.shuffle.partitions`, AQE `auto`, task count overhead | Right-size shuffle partitions; recognize the tiny-task overhead pattern |
| 5 | Name drift between producer and consumer; missing `depends_on` edge | Fix task dependency and ordering bugs; triage `TABLE_OR_VIEW_NOT_FOUND` in multi-task jobs |
| 6 | Cluster event log; notebook-scoped `%pip` | Diagnose cluster startup failures and library conflicts |

You can use the same method on any broken job. The run history shows when the failures started, the DAG
shows where, and the first line of the error shows what failed. When a job is slow or returns too many rows,
the stage metrics show why. Fix the first failure, then run the downstream tasks again. **Repair run**
reruns only the failed tasks and the tasks that depend on them.

## Cleanup

Run the cell to drop the three `ts_*` tables of this lab.

In [0]:
for _t in [RAW_TABLE, ENRICHED_TABLE, GOLD_TABLE]:
    spark.sql(f"DROP TABLE IF EXISTS {_t}")
print("Lab cleanup complete")

← <a href="$../day3/lab/lab_troubleshooting">Lab — Troubleshooting</a> | **<a href="$../..">Course folder (README)</a>**